# StockSense: content-based recommendations, step by step

This notebook walks through the recommender behind the dashboard's **Suggested for you** list,
step by step. Every step calls the same functions the API uses
(`backend/app/recommender/content_based.py`), so the results here are exactly what a signed-in
user sees.

| Section | Technique | What it gives the investor |
| --- | --- | --- |
| 1. Data understanding | Flatten the nested `risk` object; `themes` are already lists | A clean catalogue of 90 stocks and ETFs |
| 2. Structured attributes | One-hot / multi-hot features, min-max scaling, nearest neighbours | *"Similar to NVDA, which you follow"* |
| 3. Text descriptions | TF-IDF, cosine similarity, feature combination | *"Matches your interest in Technology"* |
| 4–5. Profile and ranking | Interests as queries, followed tickers, risk fit | A ranked list, each with its reasons |
| 6. Evaluation | Offline checks | Evidence the recommender does what it claims |

The content-based pipeline has three parts: a **content analyser** (item profiles), a **profile
learner** (what the investor likes) and a **filtering component** (rank items against the profile).

> Suggestions describe how much an asset *resembles* a profile. They are starting points for
> research, not forecasts and not financial advice.

**What:** Import the libraries and the recommender module from the backend.

**Why:** The notebook reuses the production code instead of re-implementing it, so the walkthrough
and the app cannot drift apart. `REPO` resolves the repository root whether the notebook is started
from `notebooks/` or from the root.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "backend"))

from app.core.onboarding import INTERESTS, RISK_LEVELS
from app.recommender.content_based import (
    FOLLOW_WEIGHT, INTEREST_WEIGHT, MIN_FOLLOW_SIMILARITY, MIN_INTEREST_SIMILARITY, RISK_FEATURES, RISK_WEIGHT,
    ContentBasedRecommender, attribute_matrix, interest_document, item_document, risk_fit, volatility_limit,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

# 1. Data understanding

The items are the recommendable stocks and ETFs. `data/processed/asset_profiles.json` is built by
`data/build_asset_profiles.py` and loaded into the `assets` table by `backend/scripts/import_assets.py`,
so it holds the same rows the API reads.

**What:** Load the catalogue and look at one asset.

**Why:** Each asset carries descriptive content (sector, category, themes) and a nested `risk`
object with one-year numbers computed from prices: annualised volatility, beta against SPY,
maximum drawdown and dividend yield.

In [2]:
assets = json.loads((REPO / "data/processed/asset_profiles.json").read_text(encoding="utf-8"))
print(len(assets), "assets")
assets[0]

90 assets


{'ticker': 'NVDA',
 'name': 'NVIDIA Corporation',
 'asset_type': 'stock',
 'category': 'AI chips and hardware',
 'sector': 'Information Technology',
 'themes': ['artificial_intelligence',
  'semiconductors',
  'gpu',
  'data_centres'],
 'recommendation_eligible': True,
 'forecast_eligible': True,
 'history_note': None,
 'as_of': '2026-09-18',
 'risk': {'volatility_1y': 0.381541,
  'beta': 1.927985,
  'max_drawdown_1y': -0.202144,
  'dividend_yield': 0.002339,
  'history_days': 252,
  'volatility_rank': 0.6333}}

**What:** Flatten the nested `risk` object into ordinary columns with `pd.json_normalize`.

**Why:** The catalogue's JSON is already structured, so cleaning is only flattening: each risk number
becomes its own column, and `themes` stays a Python list of labels ready to be turned into features.

In [3]:
data = pd.json_normalize(assets)
data.columns = [c.removeprefix("risk.") for c in data.columns]
tickers = data["ticker"].tolist()
data[["ticker", "name", "asset_type", "category", "sector", "themes", *RISK_FEATURES]].head()

,ticker,name,asset_type,category,sector,themes,volatility_1y,beta,max_drawdown_1y,dividend_yield
0,NVDA,NVIDIA Corporation,stock,AI chips and hardware,Information Technology,"[artificial_intelligence, semiconductors, gpu, data_centres]",0.381541,1.927985,-0.202144,0.002339
1,AMD,"Advanced Micro Devices, Inc.",stock,AI chips and hardware,Information Technology,"[artificial_intelligence, semiconductors, cpu, gpu]",0.715605,3.148062,-0.277608,0.000000
2,AVGO,Broadcom Inc.,stock,AI chips and hardware,Information Technology,"[artificial_intelligence, semiconductors, networking]",0.464030,2.164280,-0.294377,0.007103
3,QCOM,Qualcomm Incorporated,stock,AI chips and hardware,Information Technology,"[semiconductors, edge_ai, wireless]",0.527842,2.030307,-0.409787,0.020369
4,MU,"Micron Technology, Inc.",stock,AI chips and hardware,Information Technology,"[semiconductors, memory, data_centres]",0.794533,3.292098,-0.390954,0.000522


**What:** Count assets by type and sector, and check for missing values.

**Why:** Nearest neighbours cannot work with missing values, but dropping an asset would hide it from
every investor. So the recommender imputes the catalogue median for similarity and gives an asset with
unknown volatility no credit for risk fit. Today every asset has full risk numbers.

In [4]:
print(data["asset_type"].value_counts().to_dict())
print("recommendable:", int(data["recommendation_eligible"].sum()),
      "| missing risk numbers:", int(data[list(RISK_FEATURES)].isna().sum().sum()))
data["sector"].value_counts()

{'stock': 80, 'etf': 10}
recommendable: 90 | missing risk numbers: 0


sector
Information Technology    23
Industrials               15
Health Care               10
Financials                 9
Consumer Discretionary     7
Communication Services     7
Materials                  5
Utilities                  4
Energy                     4
Consumer Staples           4
Diversified                2
Name: count, dtype: int64

# 2. Structured attributes and nearest neighbours

The first item profile uses **structured attributes**: one-hot sector, category and asset type,
multi-hot themes (one 0/1 column per theme) and the four risk numbers.

**What:** Build the attribute matrix with `attribute_matrix()` and show the non-zero columns for NVDA and KO.

**Why:** `MultiLabelBinarizer` turns each list into 0/1 columns, so an asset with three themes has three
1s among the theme columns. The risk numbers are already min-max scaled to [0, 1] here; the next cell
shows why that matters.

In [5]:
names, X_scaled = attribute_matrix(assets)
attributes = pd.DataFrame(X_scaled, index=tickers, columns=names)
print(attributes.shape, "(assets x attributes)")
pair = attributes.loc[["NVDA", "KO"]]
pair.loc[:, pair.any()].round(2).T

(90, 186) (assets x attributes)


,NVDA,KO
category:AI chips and hardware,1.00,0.00
category:Consumer and retail,0.00,1.00
sector:Consumer Staples,0.00,1.00
sector:Information Technology,1.00,0.00
theme:artificial_intelligence,1.00,0.00
theme:beverages,0.00,1.00
theme:consumer_staples,0.00,1.00
theme:data_centres,1.00,0.00
theme:dividends,0.00,1.00
theme:gpu,1.00,0.00


## 2b. Why scaling matters

**What:** Compare the ranges of the raw risk numbers.

**Why:** Nearest neighbours uses distances, and a feature with a wide range dominates them. Beta spans
about 4.9 while dividend yield spans 0.066, so unscaled, dividend yield would be practically invisible
and beta would decide the numeric part of every distance, for no reason other than its units.

In [6]:
raw_ranges = data[list(RISK_FEATURES)].agg(["min", "max"]).T
raw_ranges["range"] = raw_ranges["max"] - raw_ranges["min"]
raw_ranges.round(3)

,min,max,range
volatility_1y,0.129,1.146,1.017
beta,-0.537,4.360,4.897
max_drawdown_1y,-0.646,-0.079,0.567
dividend_yield,0.000,0.066,0.066


**What:** Fit `NearestNeighbors` (Euclidean distance) on the raw and the min-max scaled
attributes and compare the five neighbours of a few assets.

**Why:** Same algorithm, only the preprocessing changes, which isolates the effect of scaling.

In [7]:
n_categorical = len(names) - len(RISK_FEATURES)
X_raw = np.hstack([X_scaled[:, :n_categorical], data[list(RISK_FEATURES)].to_numpy()])

def neighbours(X, ticker, k=5):
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    _, idx = nn.kneighbors(X[[tickers.index(ticker)]])
    return ", ".join(tickers[i] for i in idx[0][1:])

pd.DataFrame(
    {t: {"raw": neighbours(X_raw, t), "min-max scaled": neighbours(X_scaled, t)} for t in ["XOM", "TSLA", "BE"]}
).T

,raw,min-max scaled
XOM,"CVX, NEE, TMUS, VZ, XLE","CVX, CCJ, NEE, ETN, NRG"
TSLA,"AMZN, NKE, HD, SBUX, CCJ","AMZN, HD, SBUX, MCD, NKE"
BE,"FPS, VRT, MU, ETN, GEV","ETN, FPS, VRT, GEV, CCJ"


**Observation:** Unscaled, the telecoms TMUS and VZ sit next to Exxon (XOM) mainly because all three
have a *negative* beta, and the uranium miner CCJ sits next to Tesla because both have a beta of about 2.2.
After scaling, energy and power companies replace the telecoms, and consumer companies replace CCJ.
The effect is moderate because the many 0/1 columns already carry most of the distance, but the
lesson holds: **scale before comparing**.

## 2c. Cosine similarity: "similar to what you follow"

**What:** Compute cosine similarity between every pair of assets on the scaled attributes, then list
the closest five for a few assets.

**Why:** The recommender needs a similarity score between 0 and 1, not a distance, so it uses cosine
similarity on the same scaled vectors. When an investor follows a ticker, assets above
`MIN_FOLLOW_SIMILARITY` earn the reason *"Similar to …, which you follow"*.

In [8]:
item_sim = pd.DataFrame(cosine_similarity(X_scaled), index=tickers, columns=tickers)
pd.DataFrame({t: [f"{n} ({s:.2f})" for n, s in item_sim[t].drop(t).nlargest(5).items()] for t in ["NVDA", "KO", "JPM", "CEG"]})

,NVDA,KO,JPM,CEG
0,AMD (0.87),PG (0.71),BAC (0.85),VST (1.00)
1,MRVL (0.85),COST (0.70),MS (0.71),NRG (0.84)
2,AVGO (0.80),WMT (0.70),GS (0.71),CCJ (0.69)
3,MU (0.78),VZ (0.43),V (0.70),NEE (0.67)
4,ASML (0.67),SBUX (0.43),AXP (0.70),ETN (0.54)


**What:** Check how selective the follow threshold is.

**Why:** A threshold that most pairs pass would make "similar to" meaningless. At 0.5 only a small
share of all pairs qualifies, typically the same sub-industry.

In [9]:
pairs = item_sim.to_numpy()[~np.eye(len(tickers), dtype=bool)]
print(f"{(pairs >= MIN_FOLLOW_SIMILARITY).mean():.1%} of asset pairs reach a similarity of {MIN_FOLLOW_SIMILARITY}")

7.3% of asset pairs reach a similarity of 0.5


# 3. Text descriptions and TF-IDF

The second item profile is a **description**: the category words, sector and themes joined into one
document. Combining several fields into one text like this is called *feature combination*.

**What:** Show the documents TF-IDF reads for a few assets.

**Why:** Three deliberate choices, each found by checking results on this catalogue:
1. **Sectors and themes stay single tokens** (`renewable_energy`, `communication_services`), so each is matched as one concept.
2. **The free-text category is split into words**, so *"AI power energy and grid"* still relates grid companies to energy.
3. **Company names are left out**: *Applied Materials* is a chip-equipment maker, not a Materials company.

In [10]:
data["document"] = [item_document(a) for a in assets]
data.set_index("ticker").loc[["NVDA", "FSLR", "XOM", "XLV"], ["document"]]

,document
ticker,
NVDA,AI chips and hardware Information_Technology artificial_intelligence semiconductors gpu data_centres
FSLR,AI power energy and grid Information_Technology solar_energy renewable_energy energy_transition
XOM,AI power energy and grid Energy oil_and_gas integrated_energy traditional_energy
XLV,Healthcare ETF Health_Care healthcare sector_etf diversification


**What:** Fit the recommender, which fits `TfidfVectorizer(stop_words="english")` on the 90 documents,
and list each asset's highest-weighted terms.

**Why:** Every term is kept (no `min_df` or `max_df` cut-off): in a 90-asset catalogue a theme
such as `hydrogen` may belong to a single asset and still be the clearest
signal for an interest. IDF already gives rare, specific terms (`gpu`) more weight than common ones
(`information_technology`).

In [11]:
recommender = ContentBasedRecommender(assets)
vocab = recommender.vectorizer.get_feature_names_out()
print(recommender.descriptions.shape, "(assets x terms), sparse")

def top_terms(ticker, k=5):
    row = recommender.descriptions[tickers.index(ticker)].toarray()[0]
    return [f"{vocab[i]} ({row[i]:.2f})" for i in row.argsort()[::-1][:k] if row[i] > 0]

pd.DataFrame({t: pd.Series(top_terms(t)) for t in ["NVDA", "FSLR", "XOM", "XLV"]}).fillna("")

(90, 172) (assets x terms), sparse


,NVDA,FSLR,XOM,XLV
0,gpu (0.49),solar_energy (0.49),energy (0.52),healthcare (0.67)
1,data_centres (0.41),energy_transition (0.49),traditional_energy (0.42),sector_etf (0.40)
2,artificial_intelligence (0.35),renewable_energy (0.42),integrated_energy (0.42),diversification (0.40)
3,semiconductors (0.34),power (0.28),oil_and_gas (0.42),etf (0.34)
4,chips (0.34),grid (0.28),power (0.27),health_care (0.34)


## 3b. Why the tokens matter: a naive version

**What:** Build a naive TF-IDF that splits every theme into words, keeps company names and adds the
interest's display label to the query, then compare its top matches for *Clean Energy* with ours.

**Why:** It shows the main limitation of keyword matching in a concrete case.

In [12]:
def naive_document(asset):
    return " ".join([asset["name"], asset["category"], asset["sector"], *(t.replace("_", " ") for t in asset["themes"])])

clean = next(i for i in INTERESTS if i["key"] == "clean_energy")
naive_query = " ".join([clean["label"], *clean["sectors"], *(t.replace("_", " ") for t in clean["themes"])])

naive = TfidfVectorizer(stop_words="english")
naive_matrix = naive.fit_transform([naive_document(a) for a in assets])
naive_scores = pd.Series(cosine_similarity(naive.transform([naive_query]), naive_matrix)[0], index=tickers)
ours =pd.Series(recommender.interest_similarity(["clean_energy"])[0], index=tickers)

def top(scores, k=8):
    return [f"{t} ({s:.2f})" for t, s in scores.nlargest(k).items()]

pd.DataFrame({"naive (words, names, label)": top(naive_scores), "StockSense (tokens)": top(ours)})

,"naive (words, names, label)",StockSense (tokens)
0,FSLR (0.77),FSLR (0.60)
1,CVX (0.48),GRID (0.46)
2,NEE (0.46),PWR (0.30)
3,GRID (0.46),GEV (0.28)
4,XOM (0.46),APD (0.20)
5,XLE (0.43),NEE (0.14)
6,CCJ (0.40),FPS (0.14)
7,NRG (0.39),FCX (0.13)


**Observation:** The naive version ranks the oil majors CVX and XOM near the top for *Clean Energy*,
because "energy" appears in "renewable energy", "traditional energy" and the label itself. Keeping each
theme as one token means `traditional_energy` no longer matches `renewable_energy`, and only genuine
clean-energy names remain. Embeddings (section 7) are the other way to tackle this.

# 4. The investor profile (profile learner)

Onboarding stores four answers: **interests**, **followed tickers**, a **risk level** and preferred
**asset types**. Each becomes something we can compare with an item profile:

| Profile answer | Becomes |
| --- | --- |
| Interests | TF-IDF query documents, compared with asset descriptions |
| Followed tickers | Points in the attribute space, compared with every other asset |
| Risk level | A volatility limit shared with the rule engine |
| Asset types | A filter (stocks, ETFs or both) |

**What:** Show the query document for each interest.

**Why:** An interest is *defined* in `backend/app/core/onboarding.py` by sectors and themes, the same
definition the profile page uses to count assets. That definition is the query. The display label is
left out because "Clean Energy" would add the word "energy" again.

In [13]:
pd.DataFrame({"interest": [i["label"] for i in INTERESTS], "query document": [interest_document(i["key"]) for i in INTERESTS]})

,interest,query document
0,Technology,Information_Technology
1,Healthcare,Health_Care
2,Financials,Financials
3,Energy,Energy Utilities
4,Consumer,Consumer_Discretionary Consumer_Staples
5,Industrials,Industrials
6,Clean Energy,renewable_energy solar_energy energy_transition grid_infrastructure electrification hydrogen
7,Communication,Communication_Services
8,Materials,Materials


**What:** Cosine similarity between every interest query and every asset description, showing the
best matches above `MIN_INTEREST_SIMILARITY`.

**Why:** Unlike a yes/no rule, the score is graded. A description concentrated on the interest's terms
scores higher (XLF for Financials, XLI for Industrials), and an asset that matches several of an
interest's themes (GRID for Clean Energy) beats one that matches a single theme (APD, through `hydrogen`).

In [14]:
keys = [i["key"] for i in INTERESTS]
by_interest = pd.DataFrame(recommender.interest_similarity(keys), index=[i["label"] for i in INTERESTS], columns=tickers)
by_interest.apply(
    lambda row: ", ".join(f"{t} {s:.2f}" for t, s in row[row >= MIN_INTEREST_SIMILARITY].nlargest(7).items()), axis=1
).to_frame(f"best matches (cosine >= {MIN_INTEREST_SIMILARITY})")

,best matches (cosine >= 0.1)
Technology,"SOXX 0.28, AVGO 0.28, IGV 0.28, AMAT 0.27, QCOM 0.27, MU 0.27, MSFT 0.27"
Healthcare,"AMGN 0.43, JNJ 0.38, PFE 0.38, ABBV 0.36, MRK 0.36, LLY 0.35, ISRG 0.34"
Financials,"XLF 0.68, V 0.36, MA 0.36, JPM 0.36, BAC 0.35, GS 0.35, AXP 0.34"
Energy,"NEE 0.72, CEG 0.51, VST 0.51, NRG 0.50, XLE 0.46, CCJ 0.31, XOM 0.30"
Consumer,"PG 0.54, KO 0.53, WMT 0.53, COST 0.51, XLY 0.42, AMZN 0.26, SBUX 0.25"
Industrials,"XLI 0.58, BA 0.52, CAT 0.51, DE 0.51, UNP 0.50, UPS 0.50, RTX 0.48"
Clean Energy,"FSLR 0.60, GRID 0.46, PWR 0.30, GEV 0.28, APD 0.20, NEE 0.14, FPS 0.14"
Communication,"GOOGL 0.37, META 0.37, TMUS 0.36, VZ 0.36, NFLX 0.35, DIS 0.35, CMCSA 0.33"
Materials,"LIN 0.72, NUE 0.71, FCX 0.70, APD 0.69, ECL 0.67"


## 4b. Risk level

**What:** Show each risk level's one-year volatility limit (from the rule engine) and how many assets
fall within it.

**Why:** Only two assets are calm enough for *Very Conservative*. A hard filter would leave such an
investor with almost nothing in their interests, so risk is a **soft** signal: full credit up to the
limit, falling linearly to zero at twice the limit, and the reason text says when an asset is above it.

In [15]:
limits = pd.DataFrame({"risk level": [r["label"] for r in RISK_LEVELS],
                       "volatility limit": [volatility_limit(r["key"]) for r in RISK_LEVELS]})
limits["assets within limit"] = [int((data["volatility_1y"] <= v).sum()) for v in limits["volatility limit"]]
limits

,risk level,volatility limit,assets within limit
0,Very Conservative,0.15,2
1,Conservative,0.20,11
2,Moderate,0.35,52
3,Growth,0.45,63
4,Aggressive,0.55,76


In [16]:
pd.DataFrame(
    {f"volatility {v:.0%}": {r["label"]: risk_fit(v, r["key"]) for r in RISK_LEVELS} for v in [0.15, 0.25, 0.40, 0.60]}
).round(2)

,volatility 15%,volatility 25%,volatility 40%,volatility 60%
Very Conservative,1.0,0.33,0.00,0.00
Conservative,1.0,0.75,0.00,0.00
Moderate,1.0,1.00,0.86,0.29
Growth,1.0,1.00,1.00,0.67
Aggressive,1.0,1.00,1.00,0.91


# 5. Putting it together: `recommend()`

For each asset the investor does not already follow, and of an allowed asset type:

1. **Interest score**: best cosine similarity to any chosen interest (description space).
2. **Follow score**: best cosine similarity to any followed ticker (attribute space).
3. **Risk fit**: 0 to 1, as above.
4. The asset is suggested only if it has at least one *content* reason: an interest score of at least
   `MIN_INTEREST_SIMILARITY` or a follow score of at least `MIN_FOLLOW_SIMILARITY`.
5. **Score** = (`INTEREST_WEIGHT` x interest + `FOLLOW_WEIGHT` x follow + `RISK_WEIGHT` x risk fit) / total weight.
   Without followed tickers the follow term drops out.

The score only orders the list. Each suggestion carries its reasons, which is what the dashboard shows.

In [17]:
print(f"weights: interest {INTEREST_WEIGHT}, follow {FOLLOW_WEIGHT}, risk {RISK_WEIGHT} | "
      f"thresholds: interest {MIN_INTEREST_SIMILARITY}, follow {MIN_FOLLOW_SIMILARITY}")

def show(**profile):
    recs = recommender.recommend(**profile, limit=8)
    return pd.DataFrame([{"ticker": r.ticker, "type": r.asset_type, "score": r.score, "reasons": " | ".join(r.reasons)} for r in recs])

weights: interest 0.5, follow 0.3, risk 0.2 | thresholds: interest 0.1, follow 0.5


**Persona 1:** a conservative investor interested in Healthcare and Consumer who follows Coca-Cola (KO).

**What to notice:** KO's closest consumer-staples peers (PG, COST, WMT) come first because they both match
an interest and resemble a followed ticker. Healthcare names follow, ordered partly by how calm they are
(JNJ at 19% before PFE at 23% and AMGN at 28%).

In [18]:
show(interests=["healthcare", "consumer"], followed_tickers=["KO"], risk_level="conservative")

,ticker,type,score,reasons
0,PG,stock,0.6854,"Matches your interest in Consumer | Similar to KO, which you follow | Yearly price swings of 20% fit your Conservative risk level (up to..."
1,COST,stock,0.6660,"Matches your interest in Consumer | Similar to KO, which you follow | Yearly price swings of 20% fit your Conservative risk level (up to..."
2,WMT,stock,0.6162,"Matches your interest in Consumer | Similar to KO, which you follow | Yearly price swings of 26% are above your Conservative risk level ..."
3,JNJ,stock,0.4800,Matches your interest in Healthcare | Yearly price swings of 19% fit your Conservative risk level (up to 20%)
4,PFE,stock,0.4513,"Matches your interest in Healthcare | Yearly price swings of 23% are above your Conservative risk level (up to 20%), which lowers its rank"
5,XLY,etf,0.4500,Matches your interest in Consumer | Yearly price swings of 19% fit your Conservative risk level (up to 20%)
6,MCD,stock,0.4454,Matches your interest in Consumer | Yearly price swings of 19% fit your Conservative risk level (up to 20%)
7,AMGN,stock,0.4330,"Matches your interest in Healthcare | Yearly price swings of 28% are above your Conservative risk level (up to 20%), which lowers its rank"


**Persona 2:** a growth investor interested in Technology who follows NVDA and MSFT.

**What to notice:** each suggestion names which followed ticker it resembles. Close matches above the
45% Growth limit still appear, but they say so and lose part of their score.

In [19]:
show(interests=["technology"], followed_tickers=["NVDA", "MSFT"], risk_level="growth")

,ticker,type,score,reasons
0,ADBE,stock,0.5739,"Matches your interest in Technology | Similar to MSFT, which you follow | Yearly price swings of 41% fit your Growth risk level (up to 45%)"
1,AVGO,stock,0.5732,"Matches your interest in Technology | Similar to NVDA, which you follow | Yearly price swings of 46% are above your Growth risk level (u..."
2,CRM,stock,0.5712,"Matches your interest in Technology | Similar to MSFT, which you follow | Yearly price swings of 47% are above your Growth risk level (u..."
3,TSM,stock,0.5257,"Matches your interest in Technology | Similar to NVDA, which you follow | Yearly price swings of 40% fit your Growth risk level (up to 45%)"
4,NOW,stock,0.5242,"Matches your interest in Technology | Similar to MSFT, which you follow | Yearly price swings of 57% are above your Growth risk level (u..."
5,ASML,stock,0.5212,"Matches your interest in Technology | Similar to NVDA, which you follow | Yearly price swings of 47% are above your Growth risk level (u..."
6,ORCL,stock,0.5211,"Matches your interest in Technology | Similar to MSFT, which you follow | Yearly price swings of 57% are above your Growth risk level (u..."
7,IGV,etf,0.4983,"Matches your interest in Technology | Similar to MSFT, which you follow | Yearly price swings of 32% fit your Growth risk level (up to 45%)"


**Persona 3:** a moderate investor interested in Clean Energy and Energy who wants ETFs only.

**What to notice:** the asset-type filter leaves few candidates, and the list is short rather than padded
with unrelated ETFs. Every suggestion must have a content reason.

In [20]:
show(interests=["clean_energy", "energy"], risk_level="moderate", asset_types="etf")

,ticker,type,score,reasons
0,GRID,etf,0.6175,Matches your interest in Clean Energy | Yearly price swings of 24% fit your Moderate risk level (up to 35%)
1,XLE,etf,0.6149,Matches your interest in Energy | Yearly price swings of 22% fit your Moderate risk level (up to 35%)


**Persona 4:** a very conservative investor interested in Financials who follows nothing.

**What to notice:** XLF is the only financial asset within the 15% limit, so it leads clearly. The
individual banks still appear, flagged as above the investor's risk level.

In [21]:
show(interests=["financials"], risk_level="very_conservative")

,ticker,type,score,reasons
0,XLF,etf,0.7720,Matches your interest in Financials | Yearly price swings of 15% fit your Very Conservative risk level (up to 15%)
1,V,stock,0.4069,"Matches your interest in Financials | Yearly price swings of 22% are above your Very Conservative risk level (up to 15%), which lowers i..."
2,JPM,stock,0.4050,"Matches your interest in Financials | Yearly price swings of 22% are above your Very Conservative risk level (up to 15%), which lowers i..."
3,BAC,stock,0.4008,"Matches your interest in Financials | Yearly price swings of 22% are above your Very Conservative risk level (up to 15%), which lowers i..."
4,MA,stock,0.4007,"Matches your interest in Financials | Yearly price swings of 22% are above your Very Conservative risk level (up to 15%), which lowers i..."
5,AXP,stock,0.3017,"Matches your interest in Financials | Yearly price swings of 27% are above your Very Conservative risk level (up to 15%), which lowers i..."
6,BLK,stock,0.2846,"Matches your interest in Financials | Yearly price swings of 27% are above your Very Conservative risk level (up to 15%), which lowers i..."
7,MS,stock,0.2756,"Matches your interest in Financials | Yearly price swings of 28% are above your Very Conservative risk level (up to 15%), which lowers i..."


# 6. Evaluation

There is no interaction data yet (no clicks, follows over time or ratings), so accuracy metrics against
user behaviour are not possible. Instead, three offline checks of whether the recommender does what it
claims.

## 6a. Agreement with the onboarding definition of each interest

**What:** For each interest, compare the TF-IDF matches with the yes/no rule the profile page uses
(*sector in the interest's sectors, or any shared theme*).

**Why:** Recall against the rule should be 1: nothing the profile page promises should be missed.
Matches the rule does not contain are the graded extension of TF-IDF. They are listed so you can judge
whether they make sense.

In [22]:
def rule_matches(interest):
    return {a["ticker"] for a in assets if a["sector"] in interest["sectors"] or set(a["themes"]) & set(interest["themes"])}

rows = []
for interest in INTERESTS:
    tfidf = set(by_interest.columns[by_interest.loc[interest["label"]] >= MIN_INTEREST_SIMILARITY])
    rule = rule_matches(interest)
    rows.append({"interest": interest["label"], "TF-IDF": len(tfidf), "rule": len(rule),
                 "recall vs rule": len(tfidf & rule) / len(rule), "precision vs rule": len(tfidf & rule) / len(tfidf),
                 "TF-IDF only": ", ".join(sorted(tfidf - rule))})
pd.DataFrame(rows).round(2)

,interest,TF-IDF,rule,recall vs rule,precision vs rule,TF-IDF only
0,Technology,23,23,1.0,1.00,
1,Healthcare,10,10,1.0,1.00,
2,Financials,9,9,1.0,1.00,
3,Energy,15,8,1.0,0.53,"BE, ETN, FPS, FSLR, GEV, PWR, VRT"
4,Consumer,11,11,1.0,1.00,
5,Industrials,15,15,1.0,1.00,
6,Clean Energy,8,8,1.0,1.00,
7,Communication,7,7,1.0,1.00,
8,Materials,5,5,1.0,1.00,


**Observation:** Every asset the rule assigns to an interest is found. The only extras are for *Energy*:
grid, power-equipment and solar companies whose category is *"AI power energy and grid"*. They are
energy-related and rank below the sector's own assets, which is the kind of graded match a yes/no rule
cannot express.

## 6b. Does the risk signal move calmer assets up?

**What:** For every single-interest profile at each risk level, compare the share of assets within the
volatility limit among *all* matches with the share among the *top 5*.

**Why:** If the risk fit works, the top of the list should be calmer than the pool it was drawn from.

In [23]:
volatility = data.set_index("ticker")["volatility_1y"]
rows = []
for level in RISK_LEVELS:
    limit = volatility_limit(level["key"])
    pool, top5 = [], []
    for interest in INTERESTS:
        recs = recommender.recommend(interests=[interest["key"]], risk_level=level["key"], limit=len(assets))
        vols = [volatility[r.ticker] for r in recs]
        pool += vols
        top5 += vols[:5]
    rows.append({"risk level": level["label"], "limit": limit,
                 "within limit: all matches": np.mean(np.array(pool) <= limit),
                 "within limit: top 5": np.mean(np.array(top5) <= limit)})
pd.DataFrame(rows).round(2)

,risk level,limit,within limit: all matches,within limit: top 5
0,Very Conservative,0.15,0.01,0.02
1,Conservative,0.20,0.10,0.20
2,Moderate,0.35,0.51,0.80
3,Growth,0.45,0.65,0.82
4,Aggressive,0.55,0.83,1.00


**Observation:** At every risk level the top 5 are calmer than the pool they come from. For
*Very Conservative* the pool itself has almost nothing under 15% (only SPY and XLF in the whole
catalogue), so suggestions above the limit are unavoidable. They are labelled as such rather than hidden.

## 6c. Catalogue coverage

**What:** Count how many assets appear in at least one top-10 across every single-interest profile
(9 interests x 5 risk levels x 3 asset-type choices, nothing followed).

**Why:** Content-based recommenders can concentrate on a few items. Assets that never appear point to
gaps between the interest definitions and the catalogue.

In [24]:
seen = set()
for interest in INTERESTS:
    for level in RISK_LEVELS:
        for kind in ("both", "stock", "etf"):
            seen |= {r.ticker for r in recommender.recommend(interests=[interest["key"]], risk_level=level["key"], asset_types=kind)}
print(f"{len(seen)} of {len(tickers)} assets appear in at least one top-10")
print("never suggested from interests alone:", ", ".join(sorted(set(tickers) - seen)))

80 of 90 assets appear in at least one top-10
never suggested from interests alone: AIQ, AMD, BE, CRWD, DDOG, MRVL, PLTR, SNOW, SPY, VRT


**Observation:** Two groups never reach a top-10 from interests alone:
- **SPY and AIQ** have the sector *Diversified*, which no interest maps to.
- **The most volatile tech and power names** (AMD, MRVL, DDOG, SNOW, PLTR, CRWD, BE, VRT). Technology
  matches 23 assets with almost equal interest scores, so risk fit decides the order, and even at the
  *Aggressive* limit (55%) these are outranked by calmer assets. They still appear for investors who follow
  a similar ticker, through the follow signal.

# 7. Limitations and next steps

- **No behaviour data yet.** Content-based filtering handles new users well (no cold start), but it
  cannot learn from what similar investors explored. A hybrid with collaborative filtering becomes
  possible once interactions are recorded.
- **Over-specialisation.** An NVDA follower mostly sees other chip makers. A diversity re-ranking such
  as maximal marginal relevance could mix in related but different assets.
- **Coverage gaps (6c).** SPY and AIQ match no interest, and the most volatile names in a crowded
  interest are always outranked; both reach users only through followed tickers. A "broad market"
  interest, or more themes per technology asset, would spread suggestions further.
- **No learned embeddings (such as Doc2Vec).** Embedding models need many thousands of documents to
  learn from, and 90 short descriptions are far too few. The SEC filings and news text
  collected under `data/` could support embeddings, but generating them is a separate phase that has
  not been authorised yet.
- **Risk is backward-looking.** One-year volatility describes the past year only. The limits are shared
  with the rule engine so both parts of StockSense use one definition.
- **Hand-calibrated constants.** The weights and thresholds were checked against this catalogue (sections
  3b, 6a). Re-check them when the catalogue grows; `backend/tests/test_recommender.py` guards the key
  behaviours, including the clean-energy vs. oil case.